# 7 — Out-of-Fold Predictions for Stacking

Writes one prediction per product, each from a model that never saw that
product, in the shape the deployed model produces. The file is what a
second-stage model — the logistic regression that combines this model
with the neural network — is fitted on.

Why not a single held-out test set: it gives the second stage only a
slice of the catalogue to learn from, and the slice is thinnest exactly
where it matters — the products the two models disagree on. Folding
gives it every labelled product.

Two things have to hold for the result to be usable, and both are
settings in this notebook:

1. **The folds must match the neural network's.** They are assigned by a
   hash of the product identifier, so both projects reproduce them
   independently. `N_FOLDS`, `FOLD_SEED` and `FOLD_KEY_COL` must be the
   same in both.
2. **The model settings must match the model you deploy.** A second stage
   fitted on one model's confidence and fed another's is miscalibrated
   from the first row.

Notebook 3 also folds the catalogue, for choosing thresholds. It is left
as it was: its folds are stratified, which is right for that job and
cannot be reproduced by another project.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import json
import os
import sys

REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

import numpy as np
import pandas as pd

from product_classifier import PipelineConfig, data as data_mod, paths
from product_classifier.gridsearch import apply_saved_params
from product_classifier.crossval import out_of_fold_predictions, oof_table

---
## 1. Config

The same settings cell as notebook 3. **Check it against the model you
deploy** — vocabulary size, n-grams, `binary`, `alpha`, `min_path_count`
and above all `prediction_mode`. `joint_path` and `conditional_path`
report confidence on different scales, so predictions from one cannot
stand in for the other.

In [ ]:
cfg = PipelineConfig()

cfg.data.path = paths.data_path("cleaned_data.txt")
cfg.data.file_format = "csv"
cfg.data.delimiter = "|"
cfg.data.text_col = "FullDesc"
cfg.data.id_col = "Vallen ID"
cfg.data.level_columns = [
    "Level 1 ID",
    "Level 2 ID",
    "Level 3 ID",
    "Level 4 ID",
]

cfg.model.vectorizer.max_features = 75_000
cfg.model.vectorizer.min_df = 2
cfg.split.min_path_count = 5
cfg.model.vectorizer.ngram_range = (1, 2)

cfg.model.n_levels = 4
cfg.model.nb.alpha = 0.001
cfg.model.vectorizer.max_df = 0.95
cfg.model.vectorizer.stop_words = "english"
cfg.model.vectorizer.kind = "tfidf"
cfg.model.vectorizer.sublinear_tf = True
cfg.model.vectorizer.norm = "l2"
cfg.model.vectorizer.use_idf = False
cfg.model.vectorizer.binary = True
# Fold each bigram to its sorted form, so "ball valve" and "valve ball"
# are one feature. Only bigrams; unigrams and trigrams keep their order.
cfg.model.vectorizer.unordered_bigrams = False
cfg.model.nb.fit_prior = True
cfg.model.prediction_mode = "conditional_path"


# Raw scores everywhere. Without this the prefix probabilities come back
# normalised, and a raw threshold compared against a normalised score is
# comparing two different quantities.
cfg.model.normalise_probabilities = False

PARAMS_PATH = "../outputs/best_params.json"
if os.path.exists(PARAMS_PATH):
    cfg, _record = apply_saved_params(cfg, PARAMS_PATH)
else:
    print(f"no {PARAMS_PATH} — using the defaults above. Run notebook 2 first")
    print("if you want the searched settings.")

levels = cfg.active_levels()
cfg.model

In [ ]:
df = data_mod.load_data(cfg.data)
print(f"{len(df):,} rows | predicting {levels}")

---
## 2. Folds

`FOLD_KEY_COL` is the column the fold is derived from, and the column the
two models' predictions are later joined on. It needs to be present in
both projects' data and to identify a product uniquely. `ID` is the SKU
identifier from the pull; `Vallen ID` is the ERP product number, which
can repeat across companies.

To keep products with identical descriptions in the same fold — stricter,
since a model then never sees a held-out product's exact wording — point
`FOLD_KEY_COL` at the original description column instead. Both projects
must then use that same column, and the join key stays `ID`.

In [ ]:
N_FOLDS = 5              # must match the neural-network project
FOLD_SEED = 0            # must match the neural-network project
FOLD_KEY_COL = "ID"      # must match the neural-network project

# Identifier columns carried into the output, for joining.
ID_COLS = ["ID", "Vallen ID"]

if FOLD_KEY_COL not in df.columns:
    raise KeyError(
        f"FOLD_KEY_COL={FOLD_KEY_COL!r} is not in {cfg.data.path}.\n"
        f"  Columns: {list(df.columns)}\n"
        "  Pick a column that both projects' data files carry."
    )
_key = df[FOLD_KEY_COL].astype(str).str.strip()
print(f"{FOLD_KEY_COL!r}: {_key.nunique():,} distinct values over {len(df):,} rows")
if _key.duplicated().any():
    print(f"  >> {int(_key.duplicated().sum()):,} rows repeat an earlier value. They "
          "share its fold, which is harmless,")
    print("     but the output cannot be joined one-to-one on this column.")

---
## 3. Score every product out-of-fold

`N_FOLDS` full training runs. `with_quality=True` also records how much of
each description the scoring model could read, which the deployed model
uses to refuse descriptions it recognises nothing in.

In [ ]:
prefix, actual, folds, frame = out_of_fold_predictions(
    cfg, df=df, n_folds=N_FOLDS,
    fold_method="id_hash", fold_key_col=FOLD_KEY_COL, random_state=FOLD_SEED,
    with_quality=True,
)
oof = oof_table(prefix, actual, folds, frame, levels, id_cols=ID_COLS)
print(f"\n{len(oof):,} products scored out-of-fold")
oof.head()

### A check before saving

Accuracy by fold. The folds should agree closely; one that stands apart
means its rows differ from the rest in some way worth understanding
before anything is fitted on top.

In [ ]:
rows = []
for k in sorted(oof["fold"].unique()):
    part = oof[oof["fold"] == k]
    row = {"fold": int(k), "rows": len(part)}
    for level in levels:
        row[f"{level} accuracy"] = float(
            (part[f"Predicted {level}"] == part[f"Actual {level}"]).mean())
    rows.append(row)
by_fold = pd.DataFrame(rows)
_all = {"fold": "all", "rows": len(oof)}
for level in levels:
    _all[f"{level} accuracy"] = float(
        (oof[f"Predicted {level}"] == oof[f"Actual {level}"]).mean())
pd.concat([by_fold.astype({"fold": str}), pd.DataFrame([_all])], ignore_index=True)

In [ ]:
# Where a level's confidence describes a different category from the one
# predicted. Always 100% at full depth; see oof_table() for why the
# shallower levels can differ.
pd.DataFrame([
    {"level": levels[d - 1],
     "confidence belongs to the predicted category": float(oof[f"L{d} Prefix Agrees"].mean()),
     "unreadable descriptions": float((~oof["usable"]).mean())}
    for d in range(1, len(levels) + 1)
])

---
## 4. Save

The sidecar records what produced the file. The neural-network project
writes one too, and the two are compared before the predictions are
joined.

In [ ]:
OOF_PATH = "../outputs/oof_predictions_nb.csv"
META_PATH = "../outputs/oof_predictions_nb.json"

os.makedirs("../outputs", exist_ok=True)
oof.to_csv(OOF_PATH, index=False)

meta = {
    "model": "naive_bayes",
    "n_rows": int(len(oof)),
    "n_folds": N_FOLDS,
    "fold_seed": FOLD_SEED,
    "fold_key_col": FOLD_KEY_COL,
    "fold_method": "id_hash",
    "level_columns": list(levels),
    "prediction_mode": cfg.model.prediction_mode,
    "normalise_probabilities": cfg.model.normalise_probabilities,
    "min_path_count": cfg.split.min_path_count,
    "settings": cfg.to_dict()["model"],
}
with open(META_PATH, "w", encoding="utf-8") as fh:
    json.dump(meta, fh, indent=2, default=str)

print(f"wrote {len(oof):,} rows to {OOF_PATH}")
print(f"wrote {META_PATH}")
print(f"columns: {list(oof.columns)}")